По базе машин с ЮЛЫ данным обучите модель для предсказания цен на машины.

1. Создайте обучающую, тестовую и проверочную выборки.

2. Оцените качество работы созданной сети, определив средний процент ошибки на проверочной выборке. (Для этого потребуется привести предсказанные моделью значения к первоначальному диапазону цен.)  

3. Подсчитайте ошибку на каждом примере тестовой выборки и суммарный процент ошибки.


Рекомендации:
- в качестве ошибки рекомендуется использовать среднеквадратическую ошибку (mse).
- метрику для данной задачи можно не использовать.
- последний слой модели должен иметь 1 нейрон.
- суммарный процент ошибки = средний модуль ошибки (MAE) / среднюю цену машины. Например, если средняя цена машины 560.000 р, а средняя ошибка 56.000р, то процент ошибки равен 10%.


In [ ]:
import pandas as pd
import numpy as np
import gdown

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# =========================
# 1. ЗАГРУЗКА ДАННЫХ
# =========================
gdown.download(
    'https://storage.yandexcloud.net/aiueducation/Content/base/l10/cars_new.csv',
    None,
    quiet=True
)

df = pd.read_csv('cars_new.csv')


In [ ]:
y = df['price'].values.reshape(-1, 1)

X = df.drop('price', axis=1)

cat_cols = ['mark', 'model', 'body', 'kpp', 'fuel']

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X.values,
    y,
    test_size=0.3,
    random_state=42
)
# Делим временную выборку пополам:
# 15%  validation
# 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)


In [ ]:
# Нормализация входных признаков
x_scaler = StandardScaler()

X_train = x_scaler.fit_transform(X_train)
X_val = x_scaler.transform(X_val)
X_test = x_scaler.transform(X_test)

# Масштабируем цены автомобилей
# для более стабильного обучения
y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(y_train)
y_val_scaled = y_scaler.transform(y_val)
y_test_scaled = y_scaler.transform(y_test)

In [ ]:
model = Sequential([
    layers.Dense(256, activation='relu',input_shape=(X_train.shape[1],)),

    layers.BatchNormalization(),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),

    layers.Dense(64, activation='relu'),

    layers.Dense(32, activation='relu'),

    layers.Dense(1)
])

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='mse'
)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    verbose=1
)

In [ ]:
history = model.fit(
    X_train,
    y_train_scaled,
    validation_data=(X_val, y_val_scaled),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Epoch 1/30
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.4140 - val_loss: 0.4398 - learning_rate: 0.0010
Epoch 2/30
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 24s 16ms/step - loss: 0.2333 - val_loss: 0.3923 - learning_rate: 0.0010
Epoch 3/30
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - loss: 0.1865 - val_loss: 0.4010 - learning_rate: 0.0010
Epoch 4/30
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - loss: 0.1705 - val_loss: 0.3942 - learning_rate: 0.0010
Epoch 5/30
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 0.1552 - val_loss: 0.3259 - learning_rate: 0.0010
Epoch 6/30
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 24s 16ms/step - loss: 0.1274 - val_loss: 0.9368 - learning_rate: 0.0010
Epoch 7/30
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 40s 15ms/step - loss: 0.1237 - val_loss: 1.4395 - learning_rate: 0.0010
Epoch 8/30
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - loss: 0.1164 - val_loss: 1.1911 - learning_rate: 0.0010
Epoch 9/30
1534/1534 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - loss: 0.1197 - val_loss: 0.708

In [ ]:
y_pred_scaled = model.predict(X_test)

# Возвращаем реальные цены
y_pred = y_scaler.inverse_transform(y_pred_scaled).squeeze()
y_true = y_test.squeeze()


329/329 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step


In [ ]:
mae = np.mean(np.abs(y_true - y_pred))

mean_price = np.mean(y_true)

percent_error = mae / mean_price * 100
print(f"MAE: {mae:.2f}")
print(f"Средняя цена: {mean_price:.2f}")
print(f"Суммарный процент ошибки: {percent_error:.2f}%")

MAE: 98772.90
Средняя цена: 523842.48
Суммарный процент ошибки: 18.86%


In [ ]:
errors = np.abs(y_true - y_pred)
percent_errors = errors / y_true * 100

for i in range(10):
    print(
        f"True: {y_true[i]:.0f} | "
        f"Pred: {y_pred[i]:.0f} | "
        f"Abs error: {errors[i]:.0f} "
        f"({percent_errors[i]:.2f}%)"
    )

True: 370000 | Pred: 460494 | Abs error: 90494 (24.46%)
True: 100000 | Pred: 106752 | Abs error: 6752 (6.75%)
True: 295000 | Pred: 316296 | Abs error: 21296 (7.22%)
True: 140000 | Pred: 220409 | Abs error: 80409 (57.44%)
True: 1150000 | Pred: 1014922 | Abs error: 135078 (11.75%)
True: 1499000 | Pred: 1341066 | Abs error: 157934 (10.54%)
True: 165000 | Pred: 188717 | Abs error: 23717 (14.37%)
True: 190000 | Pred: 194525 | Abs error: 4525 (2.38%)
True: 540000 | Pred: 398203 | Abs error: 141797 (26.26%)
True: 335000 | Pred: 288202 | Abs error: 46798 (13.97%)
